In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


In [ ]:
mnist_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transform)

fashion_full = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
fashion_test = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.50MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.4MB/s]
100%|██████████| 26.4M/26.4M [00:02<00:00, 10.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 191kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.49MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 27.6MB/s]


In [ ]:
def split_70_10_20(train_dataset, test_dataset):
    total = len(train_dataset)
    train_len = int(0.7 * total)
    val_len = int(0.1 * total)
    rem = total - train_len - val_len
    train_ds, val_ds, _ = random_split(train_dataset, [train_len, val_len, rem])
    return train_ds, val_ds, test_dataset

mnist_train, mnist_val, mnist_test = split_70_10_20(mnist_full, mnist_test)
fashion_train, fashion_val, fashion_test = split_70_10_20(fashion_full, fashion_test)


In [ ]:
def build_resnet(name):
    if name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    else:
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    correct,total = 0,0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast():
            out = model(x)
            loss = criterion(out,y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        correct += (out.argmax(1)==y).sum().item()
        total += y.size(0)
    return correct/total


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct,total = 0,0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1)==y).sum().item()
            total += y.size(0)
    return correct/total


In [ ]:
def make_loaders(train_ds, val_ds, test_ds, batch_size, pin_memory=False):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=pin_memory)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin_memory)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin_memory)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(mnist_train, mnist_val, mnist_test, batch_size=16, pin_memory=True)


In [ ]:
model = build_resnet("resnet18").to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()

for epoch in range(5):
    train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    val_acc = evaluate(model, val_loader)
    print(epoch, train_acc, val_acc)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 89.6MB/s]
/tmp/ipython-input-4241227372.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-2307076254.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


0 0.9658333333333333 0.9858333333333333
1 0.9826428571428572 0.9898333333333333
2 0.9869761904761905 0.9873333333333333
3 0.9896904761904762 0.9908333333333333
4 0.9920238095238095 0.9921666666666666
